# Capítulo 2: Probabilidade

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma explicação curta; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 2.1 Modelos Probabilísticos

Importa as bibliotecas usadas no capítulo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from formato import num

Simula 60, 600 e 60.000 lançamentos de um dado equilibrado e calcula a frequência relativa de cada face. A semente 42 faz o resultado sair igual ao do livro.

In [ ]:
rng = np.random.default_rng(42)

frequencias = {}
for n in [60, 600, 60_000]:
    lancamentos = rng.integers(1, 7, size=n)   # faces de 1 a 6
    contagem = pd.Series(lancamentos).value_counts(normalize=True)
    frequencias[n] = contagem.sort_index()

tabela = pd.DataFrame(frequencias)
tabela.index.name = "face"
tabela.round(3)

Desenha as três distribuições de frequência lado a lado, com a linha do modelo em 1/6. Com n pequeno as barras oscilam; com n grande ficam coladas na linha.

In [ ]:
fig, eixos = plt.subplots(1, 3, sharey=True, figsize=(9, 3.5))
for eixo, n in zip(eixos, tabela.columns):
    eixo.bar(tabela.index, tabela[n])
    eixo.axhline(1/6, color="black", linestyle="--", linewidth=1)
    eixo.set_title(f"n = {num(n, 0)}")
    eixo.set_xlabel("face")
eixos[0].set_ylabel("frequência relativa")
plt.tight_layout()
plt.show()

Enumera o espaço amostral de três builds, cada um passando (P) ou falhando (F). O `product` faz o produto cartesiano: são $2 \times 2 \times 2 = 8$ sequências.

In [ ]:
omega = ["".join(r) for r in product("PF", repeat=3)]
omega

Representa o modelo do dado como um dicionário (ponto amostral → probabilidade) e os eventos como conjuntos. A função `prob` soma as probabilidades dos pontos de um evento.

In [ ]:
modelo_dado = {face: 1/6 for face in range(1, 7)}

par = {2, 4, 6}
maior_que_3 = {4, 5, 6}

def prob(evento, modelo):
    return sum(modelo[w] for w in evento)

prob(par, modelo_dado), prob(maior_que_3, modelo_dado)

Monta o evento "exatamente dois falham". Um `set` de strings não tem ordem fixa, então o `sorted` deixa a saída sempre igual.

In [ ]:
dois_falham = {w for w in omega if w.count("F") == 2}
sorted(dois_falham)

Carrega os dados dos estados e monta o modelo do sorteio de um estado: 27 pontos amostrais, cada um com probabilidade 1/27.

In [ ]:
estado = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/estados.csv")

modelo_uf = {sigla: 1/27 for sigla in estado["Sigla"]}
len(modelo_uf)

Calcula a probabilidade de sortear um estado com mais de 10 milhões de habitantes. Com pontos equiprováveis, ela é a frequência relativa, que o pandas calcula direto com `.mean()`.

In [ ]:
grandes = set(estado.loc[estado["Populacao"] > 10_000_000, "Sigla"])
print(sorted(grandes), prob(grandes, modelo_uf))
print((estado["Populacao"] > 10_000_000).mean())

Calcula a probabilidade de sortear um estado com taxa de homicídios acima da mediana. Dá 13/27, e não 1/2: com $n = 27$ ímpar, a mediana é a taxa de um estado, que não fica acima nem abaixo dela.

In [ ]:
mediana_taxa = estado["Taxa.Homicidios"].median()
acima = set(estado.loc[estado["Taxa.Homicidios"] > mediana_taxa, "Sigla"])
len(acima), prob(acima, modelo_uf)

Troca o modelo: agora sorteamos uma **pessoa**, e a probabilidade de cada estado é a fração da população que mora nela. O evento é o mesmo conjunto de siglas, mas a probabilidade muda.

In [ ]:
modelo_pessoa = dict(zip(estado["Sigla"], estado["Populacao"] / estado["Populacao"].sum()))
prob(grandes, modelo_uf), prob(grandes, modelo_pessoa)

## 2.2 Propriedades da Probabilidade

Continua usando `estado`, `modelo_uf` e `prob` da seção 2.1. Monta o evento certo (todas as siglas) e um evento impossível (nenhum estado passa de 50 milhões de habitantes): as probabilidades são 1 e 0.

In [ ]:
todos = set(estado["Sigla"])
mais_de_50_milhoes = set(estado.loc[estado["Populacao"] > 50_000_000, "Sigla"])

prob(todos, modelo_uf), prob(mais_de_50_milhoes, modelo_uf)

Define os eventos $A$ (estado **grande**, com mais de 10 milhões de habitantes) e $B$ (**taxa alta**, com taxa de homicídios acima da mediana) e conta quantos estados cada um tem.

In [ ]:
A = set(estado.loc[estado["Populacao"] > 10_000_000, "Sigla"])
mediana_taxa = estado["Taxa.Homicidios"].median()
B = set(estado.loc[
    estado["Taxa.Homicidios"] > mediana_taxa, "Sigla"
])

len(A), len(B)

Em Python, `&` é a interseção e `|` é a união de dois conjuntos. Só um estado está nos dois eventos.

In [ ]:
sorted(A & B), len(A | B)

Monta a tabela de contingência dos dois eventos: as quatro casas e os totais.

In [ ]:
grande = estado["Sigla"].isin(A).map({True: "A: grande", False: "não grande"})
taxa_alta = estado["Sigla"].isin(B).map({True: "B: acima da mediana", False: "até a mediana"})

pd.crosstab(grande.rename(None), taxa_alta.rename(None), margins=True, margins_name="Total")

Confere a regra da adição: $P(A) + P(B) - P(A \cap B)$ dá o mesmo que a probabilidade da união calculada direto.

In [ ]:
soma_com_desconto = prob(A, modelo_uf) + prob(B, modelo_uf) - prob(A & B, modelo_uf)
direto = prob(A | B, modelo_uf)

round(soma_com_desconto, 4), round(direto, 4)

Um evento $C$ (menos de 1 milhão de habitantes) que não tem nenhum estado em comum com $A$: a interseção é o conjunto vazio, `set()`.

In [ ]:
C = set(estado.loc[estado["Populacao"] < 1_000_000, "Sigla"])
sorted(C), A & C

O complementar de $A$ é `todos - A`. A probabilidade dele bate com $1 - P(A)$, a menos de arredondamento na última casa.

In [ ]:
Ac = todos - A
len(Ac), prob(Ac, modelo_uf), 1 - prob(A, modelo_uf)

Por causa desse arredondamento, `==` diz que os dois números são diferentes, e `isclose` diz que são iguais dentro de uma tolerância.

In [ ]:
from math import isclose

prob(Ac, modelo_uf) == 1 - prob(A, modelo_uf), isclose(prob(Ac, modelo_uf), 1 - prob(A, modelo_uf))

Confere as duas leis de De Morgan com os conjuntos.

In [ ]:
Bc = todos - B
(todos - (A | B)) == (Ac & Bc), (todos - (A & B)) == (Ac | Bc)

## 2.3 Contagem e Probabilidade Clássica

Importa as funções de contagem do módulo `math`.

In [ ]:
from math import comb, factorial, perm

Lista todos os PINs de 4 dígitos com `product` e confere que há $10^4$ deles.

In [ ]:
pins = list(product(range(10), repeat=4))
print(len(pins), 10**4)
pins[:3]

Confere fatorial, permutação e combinação contra as listas geradas pelo `itertools`.

In [ ]:
from itertools import combinations, permutations

testes = ["T1", "T2", "T3", "T4", "T5"]

print(factorial(5), len(list(permutations(testes))))     # ordens
print(perm(5, 3), len(list(permutations(testes, 3))))  # 3 primeiros
print(comb(5, 3), len(list(combinations(testes, 3))))  # grupos de 3

Probabilidade de o auditor encontrar exatamente 2 PRs com bug ao sortear 4 entre 20 (5 com bug).

In [ ]:
p_dois = comb(5, 2) * comb(15, 2) / comb(20, 4)
round(p_dois, 4)

Probabilidade de nenhum PR com bug e, pelo complementar, de pelo menos um.

In [ ]:
p_nenhum = comb(15, 4) / comb(20, 4)
round(p_nenhum, 4), round(1 - p_nenhum, 4)

Simula 100.000 auditorias com semente 42 e compara as frequências relativas com as contas.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
lote = np.array([1] * 5 + [0] * 15)       # 1 = PR com bug
bugs = np.array([rng.choice(lote, size=4, replace=False).sum() for _ in range(100_000)])

round(float((bugs == 2).mean()), 4), round(float((bugs >= 1).mean()), 4)

Monta a tabela da Mega-Sena: quantas apostas simples cabem em cada volante, o preço (R\$ 6,00 por aposta simples) e a chance da sena.

In [ ]:
N = comb(60, 6)
marcados = range(6, 21)
tabela = pd.DataFrame({"apostas": [comb(k, 6) for k in marcados]},
                      index=pd.Index(marcados, name="números"))
tabela["preço (R$)"] = 6 * tabela["apostas"]
tabela["1 chance em"] = (N / tabela["apostas"]).round().astype(int)

tabela.loc[[6, 7, 8, 10, 15, 20]]

## 2.4 Probabilidade Condicional e Independência

Importa `Fraction`, que mostra probabilidades como frações exatas.

In [ ]:
from fractions import Fraction

Probabilidade de taxa alta dado que o estado é grande, $P(B \mid A)$, comparada com $P(B)$ sem informação nenhuma.

In [ ]:
p_b_dado_a = prob(A & B, modelo_uf) / prob(A, modelo_uf)
Fraction(p_b_dado_a).limit_denominator(), round(p_b_dado_a, 3), round(prob(B, modelo_uf), 3)

A tabela de contingência com `normalize="index"` dá as probabilidades condicionais de cada linha.

In [ ]:
grande = estado["Sigla"].isin(A).map({True: "A: grande", False: "não grande"})
taxa_alta = estado["Sigla"].isin(B).map({True: "B: taxa alta", False: "taxa não alta"})

pd.crosstab(grande, taxa_alta, rownames=[""], colnames=[""], normalize="index").round(3)

Enumera os 20 pares ordenados de PRs sorteados sem reposição e confere que o segundo tem bug com probabilidade 2/5.

In [ ]:
from itertools import permutations

prs = ["B1", "B2", "S1", "S2", "S3"]
pares = list(permutations(prs, 2))        # 5 x 4 = 20 pares ordenados

bug_no_segundo = [p for p in pares if p[1].startswith("B")]
len(pares), Fraction(len(bug_no_segundo), len(pares))

Testa a independência entre ser grande e ter taxa alta: compara $P(A \cap B)$ com $P(A)\,P(B)$.

In [ ]:
lado_esq = prob(A & B, modelo_uf)
lado_dir = prob(A, modelo_uf) * prob(B, modelo_uf)
round(lado_esq, 4), round(lado_dir, 4)

Confiabilidade de um sistema em série (autenticação e banco) e do banco com uma réplica em paralelo.

In [ ]:
p_auth, p_banco = 0.99, 0.98

serie = p_auth * p_banco
banco_replicado = 1 - (1 - p_banco) ** 2
com_replica = p_auth * banco_replicado

round(serie, 4), round(banco_replicado, 4), round(com_replica, 4)

## 2.5 O Teorema de Bayes

Detector de fraude: probabilidade total do alerta e probabilidade a posteriori de fraude dado o alerta.

In [ ]:
p_fraude = 0.01
p_alerta_dado_fraude = 0.99       # verossimilhança da fraude
p_alerta_dado_legitima = 0.05     # alarme falso

p_alerta = p_fraude * p_alerta_dado_fraude + (1 - p_fraude) * p_alerta_dado_legitima
p_fraude_dado_alerta = p_fraude * p_alerta_dado_fraude / p_alerta

round(p_alerta, 4), round(p_fraude_dado_alerta, 4)

Simula 100.000 transações com semente 42 e mede a fração de fraudes entre as que dispararam o alerta.

In [ ]:
rng = np.random.default_rng(42)
n = 100_000

fraude = rng.random(n) < p_fraude
alerta = np.where(fraude,
                  rng.random(n) < p_alerta_dado_fraude,
                  rng.random(n) < p_alerta_dado_legitima)

int(alerta.sum()), int((fraude & alerta).sum()), round(float((fraude & alerta).sum() / alerta.sum()), 3)

Teorema de Bayes com três hipóteses: nível do candidato dado que foi aprovado no teste técnico.

In [ ]:
priori = {"bom": 0.25, "médio": 0.50, "fraco": 0.25}
verossimilhanca = {"bom": 0.80, "médio": 0.50, "fraco": 0.20}   # P(aprovado | nível)

p_aprovado = sum(priori[c] * verossimilhanca[c] for c in priori)
posteriori = {c: round(priori[c] * verossimilhanca[c] / p_aprovado, 2) for c in priori}

print("P(aprovado) =", round(p_aprovado, 2))
for nivel, valor in posteriori.items():
    print(nivel, valor)

Atualização sequencial: a posteriori depois da primeira informação vira a priori da segunda.

In [ ]:
def bayes(priori, veross_E, veross_nao_E):
    return priori * veross_E / (priori * veross_E + (1 - priori) * veross_nao_E)

p1 = bayes(0.10, 0.20, 0.05)       # depois do aumento de latência
p2 = bayes(p1, 0.10, 0.60)         # P(B2 | E∩B1) e P(B2 | Eᶜ∩B1)
round(p1, 2), round(p2, 2)